# **Multi-Brand Marketing Campaign Performance Analysis**

In [40]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

In [41]:
final_df = pd.read_csv(r"D:\PROJECTS\Anna_Project_3\Multi-Brand_Marketing_Campaign_Performance_Analysis\CSV\final_df.csv",index_col=0)

# **Exploratory Data Analysis (EDA)**

### **Analyze campaign performance across brands**

In [42]:
analysis_2 = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

fig = px.bar(
    analysis_2,
    x='Campaign_Type',
    y='ROI',
    color='ROI',
    title='Average ROI Across Brands'
)

fig.show()

### **Identify top-performing and low-performing campaigns**

In [43]:
top_campaigns = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .nlargest(5)
    .reset_index()
)

top_campaigns

,Campaign_Type,ROI
0,Paid Ads,2.170926
1,Social Media,2.152836
2,SEO,2.150435
3,Email,2.146171
4,Influencer,2.131785


In [44]:
low_campaigns = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .nsmallest(5)
    .reset_index()
)

low_campaigns

,Campaign_Type,ROI
0,Influencer,2.131785
1,Email,2.146171
2,SEO,2.150435
3,Social Media,2.152836
4,Paid Ads,2.170926


### **Explore relationships between spend, clicks, revenue, and ROI**

In [45]:
final_df.columns

Index(['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration',
       'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions',
       'Revenue', 'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score',
       'Customer_Segment', 'Date', 'ROI_Flag'],
      dtype='str')

In [46]:
final_df[['Acquisition_Cost','Clicks','Revenue','ROI']].corr()

,Acquisition_Cost,Clicks,Revenue,ROI
Acquisition_Cost,1.000000,-0.624654,-0.623756,-0.677989
Clicks,-0.624654,1.000000,0.709879,0.580746
Revenue,-0.623756,0.709879,1.000000,0.795874
ROI,-0.677989,0.580746,0.795874,1.000000


In [47]:
corr_matrix = final_df[['Acquisition_Cost', 'Clicks', 'Revenue', 'ROI']].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Correlation Between Spend, Clicks, Revenue, and ROI'
)

fig.show()

### **Analyze channel-wise effectiveness**

In [48]:
analysis_channel = (
    final_df
    .groupby('Channel_Used')
    .agg({
        'Revenue': 'sum',
        'ROI': 'mean',
        'Clicks': 'sum',
        'Conversions': 'sum',
        'Leads': 'sum',
        'Impressions': 'sum'
    })
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

analysis_channel

,Channel_Used,Revenue,ROI,Clicks,Conversions,Leads,Impressions
74,"Google, YouTube, Email",1.956274e+08,2.463389,1845204.000,396887.250,728574.5,2.107648e+07
19,"Email, WhatsApp, Instagram",2.089846e+08,2.425872,2041032.500,441094.875,804009.0,2.385009e+07
147,"YouTube, Instagram, Email",2.135476e+08,2.404145,2141158.500,464640.375,814358.5,2.560505e+07
55,"Google, Email, Instagram",2.156475e+08,2.395129,2053972.125,455223.125,839186.5,2.374507e+07
15,"Email, Instagram, YouTube",1.988379e+08,2.388594,1929633.375,414410.750,765959.5,2.335334e+07
...,...,...,...,...,...,...,...
23,"Email, YouTube, Google",1.934908e+08,1.948601,1978978.875,400366.125,758603.0,2.333147e+07
128,"WhatsApp, YouTube, Google",1.943920e+08,1.942401,1912482.250,396147.375,738475.0,2.299289e+07
44,"Facebook, WhatsApp, Google",1.932911e+08,1.924841,1954222.875,413844.375,753057.0,2.383391e+07
24,"Email, YouTube, Instagram",1.583125e+08,1.866805,1580413.250,330353.375,619921.0,1.975000e+07


In [49]:
channel_df = final_df.copy()

channel_df['Channel_Used'] = channel_df['Channel_Used'].str.split(', ')

channel_df = channel_df.explode('Channel_Used')

analysis_channel = (
    channel_df
    .groupby('Channel_Used')
    .agg({
        'Revenue': 'sum',
        'ROI': 'mean',
        'Clicks': 'sum',
        'Conversions': 'sum',
        'Leads': 'sum',
        'Impressions': 'sum'
    })
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

analysis_channel

,Channel_Used,Revenue,ROI,Clicks,Conversions,Leads,Impressions
0,Email,2.405592e+10,2.175943,2.380212e+08,5.036417e+07,93579045.0,2.814183e+09
3,Instagram,2.412612e+10,2.169851,2.378365e+08,5.048938e+07,93528552.0,2.817075e+09
1,Facebook,2.395987e+10,2.148465,2.373479e+08,5.027888e+07,93215914.5,2.819927e+09
2,Google,2.385267e+10,2.140691,2.355146e+08,4.981124e+07,92441340.5,2.806382e+09
4,WhatsApp,2.389286e+10,2.140504,2.365846e+08,5.013085e+07,92931951.0,2.806107e+09
5,YouTube,2.382342e+10,2.140365,2.368446e+08,5.001778e+07,92839517.5,2.818644e+09


#### **Encoding**

In [50]:
final_df.select_dtypes("object").columns

C:\Users\saran\AppData\Local\Temp\ipykernel_29932\312379875.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  final_df.select_dtypes("object").columns


Index(['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Channel_Used',
       'Language', 'Customer_Segment', 'Date', 'ROI_Flag'],
      dtype='str')

In [51]:
label_columns = ['Campaign_Type', 'Target_Audience', 'Language', 'Customer_Segment']
label_encoders = {}
 
for col in label_columns:
    le = LabelEncoder()
    final_df[col] = le.fit_transform(final_df[col])
    label_encoders[col] = le          # keep this column's own fitted encoder
 
joblib.dump(label_encoders, "label_encoders.pkl")
print("Label encoders saved successfully!")

Label encoders saved successfully!


In [52]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

channel_encoded = mlb.fit_transform(
    final_df["Channel_Used"].str.split(", ")
)

channel_df = pd.DataFrame(
    channel_encoded,
    columns=mlb.classes_,
    index=final_df.index
)

final_df = pd.concat([final_df.drop(columns=["Channel_Used"]), channel_df], axis=1)

In [53]:
final_df['ROI_Flag'].value_counts()

ROI_Flag
Profit    119493
Loss       33759
Name: count, dtype: int64

In [54]:
final_df['ROI_Flag'] = final_df['ROI_Flag'].map({'Profit':0,'Loss':1})

# **Model Building**

## **Regression Model**

In [55]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVR
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [56]:
final_df.columns

Index(['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration',
       'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue',
       'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score',
       'Customer_Segment', 'Date', 'ROI_Flag', 'Email', 'Facebook', 'Google',
       'Instagram', 'WhatsApp', 'YouTube'],
      dtype='str')

In [57]:
X_r = final_df.drop(columns=['Campaign_ID','ROI','ROI_Flag','Revenue','Language','Campaign_Type','Date'])
y_r = final_df['Revenue']

In [58]:
X_train_r,X_test_r,y_train_r,y_test_r = train_test_split(X_r,y_r,test_size=0.20,random_state=42)

In [59]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_r)
X_test = scaler.transform(X_test_r)

In [60]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    metrics = {
        "Model": name,
        "r2 Score": r2_score(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds),
        "MSE": mean_squared_error(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds)
    }

    print(f"--- {name} ---")
    print(f"r2 Score : {metrics['r2 Score']:.4f}")
    print(f"MAE: {metrics['MAE']:.4f}")
    print(f"MSE: {metrics['MSE']:.4f}")
    print(f"RMSE: {metrics['RMSE']:.4f}")

    print()

    return model, preds, metrics

In [61]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 153252 entries, 0 to 153251
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       153252 non-null  str    
 1   Campaign_Type     153252 non-null  int64  
 2   Target_Audience   153252 non-null  int64  
 3   Duration          153252 non-null  float64
 4   Impressions       153252 non-null  float64
 5   Clicks            153252 non-null  float64
 6   Leads             153252 non-null  float64
 7   Conversions       153252 non-null  float64
 8   Revenue           153252 non-null  float64
 9   Acquisition_Cost  153252 non-null  float64
 10  ROI               153252 non-null  float64
 11  Language          153252 non-null  int64  
 12  Engagement_Score  153252 non-null  float64
 13  Customer_Segment  153252 non-null  int64  
 14  Date              153252 non-null  str    
 15  ROI_Flag          153252 non-null  int64  
 16  Email             153252 non-nu

In [62]:
models = {
        "Linear Regression": LinearRegression(),
        "KNN": KNeighborsRegressor(n_neighbors=3),
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        )
    }

results = []
predictions = {}
fitted_models = {}

for name, model in models.items():
    fitted_model, preds, metrics = evaluate_model(
        name, model, X_train, y_train_r, X_test, y_test_r
    )
    results.append(metrics)
    predictions[name] = preds
    fitted_models[name] = fitted_model

--- Linear Regression ---
r2 Score : 0.7148
MAE: 141039.1535
MSE: 37534426907.1554
RMSE: 193738.0368

--- KNN ---
r2 Score : 0.6220
MAE: 162143.0701
MSE: 49756212648.1678
RMSE: 223061.0066

--- Decision Tree ---
r2 Score : 0.4798
MAE: 179627.8115
MSE: 68475460120.6895
RMSE: 261678.1613

--- Random Forest ---
r2 Score : 0.7386
MAE: 136046.7809
MSE: 34411209569.3466
RMSE: 185502.5864

--- Gradient Boosting ---
r2 Score : 0.7446
MAE: 134552.1147
MSE: 33620483496.2798
RMSE: 183358.8926

--- XGBoost ---
r2 Score : 0.7382
MAE: 134938.3689
MSE: 34458353995.2703
RMSE: 185629.6151



In [63]:
final_df

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,...,Engagement_Score,Customer_Segment,Date,ROI_Flag,Email,Facebook,Google,Instagram,WhatsApp,YouTube
0,NY-CMP-1000,4,0,21.000000,57804.0,6156.0,3616.0,2355.0,1370861.875,207.23,...,20.98,0,2025-04-29,0,0,0,0,0,1,1
1,NY-CMP-1001,2,2,18.000000,91801.0,3321.0,1971.0,1357.0,1046247.000,180.83,...,7.24,0,2025-04-06,0,0,0,0,0,0,1
2,NY-CMP-1002,1,4,23.000000,15536.0,2182.0,952.0,755.0,197055.000,90.60,...,25.03,0,2025-01-14,0,0,0,1,0,1,1
3,NY-CMP-1003,0,3,18.000000,88114.0,8413.0,2231.0,947.0,376906.000,249.07,...,13.15,0,2025-06-04,0,0,1,0,1,0,1
4,NY-CMP-1004,2,0,10.000000,96871.0,3743.0,2060.0,1258.0,518296.000,228.60,...,7.29,2,2024-12-29,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153247,TI-CMP-56548,1,1,15.000000,25929.0,1388.0,808.0,567.0,408240.000,258.43,...,10.66,3,2025-05-19,0,0,0,1,0,1,1
153248,TI-CMP-56549,2,1,26.000000,33252.0,2871.0,950.0,777.0,359270.000,421.81,...,12.96,2,2024-09-30,0,0,1,0,0,0,0
153249,TI-CMP-56551,3,3,20.000000,54886.0,1578.0,634.0,777.0,136514.000,436.24,...,4.66,3,2024-08-25,1,0,0,1,1,0,0
153250,TI-CMP-56552,0,1,13.000000,97954.0,11480.0,4567.0,2144.0,445952.000,78.77,...,18.57,1,2025-04-15,0,0,1,0,1,0,0


In [64]:
import pandas as pd

results_df = pd.DataFrame(results)

best_model_name = results_df.loc[
    results_df["r2 Score"].idxmax(),
    "Model"
]

best_model = fitted_models[best_model_name]

print("Best Model:", best_model_name)
print("Best r2 Score:",
      results_df["r2 Score"].max())

Best Model: Gradient Boosting
Best r2 Score: 0.7445815057800562


In [65]:
import joblib

joblib.dump(best_model, "best_Regression_model.pkl")
joblib.dump(scaler, "Regression_scaler.pkl")

print("Model saved successfully!")


Model saved successfully!


In [66]:
model_columns = X_r.columns.tolist()
joblib.dump(model_columns, "model_columns_regression.pkl")

['model_columns_regression.pkl']

## **Classification Model**

In [67]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [68]:
X_c = final_df.drop(
    columns=['Campaign_ID', 'ROI', 'ROI_Flag',
             'Language', 'Campaign_Type', 'Date']
)
y_c=final_df['ROI_Flag']

In [69]:
final_df.columns

Index(['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration',
       'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue',
       'Acquisition_Cost', 'ROI', 'Language', 'Engagement_Score',
       'Customer_Segment', 'Date', 'ROI_Flag', 'Email', 'Facebook', 'Google',
       'Instagram', 'WhatsApp', 'YouTube'],
      dtype='str')

In [70]:
X_train_c,X_test_c,y_train_c,y_test_c = train_test_split(X_c,y_c,test_size=0.20,random_state=42)

In [71]:
scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c = scaler.transform(X_test_c)

In [72]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1 Score": f1_score(y_test, preds, zero_division=0),
        "Confusion Matrix": confusion_matrix(y_test, preds)
    }

    print(f"--- {name} ---")
    print(f"Accuracy : {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall   : {metrics['Recall']:.4f}")
    print(f"F1 Score : {metrics['F1 Score']:.4f}")
    print("Confusion Matrix:")
    print(metrics["Confusion Matrix"])
    print()

    return model, preds, metrics

In [73]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    "KNN": KNeighborsClassifier(n_neighbors=3),

    "Decision Tree": DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ),

    "SVM": SVC(
        class_weight='balanced'
    ),

    "Naive Bayes": GaussianNB(),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42
    )
}

results = []
predictions = {}
fitted_models = {}

for name, model in models.items():
    fitted_model, preds, metrics = evaluate_model(
        name, model, X_train_c, y_train_c, X_test_c, y_test_c
    )
    results.append(metrics)
    predictions[name] = preds
    fitted_models[name] = fitted_model

--- Logistic Regression ---
Accuracy : 0.9540
Precision: 0.8503
Recall   : 0.9621
F1 Score : 0.9027
Confusion Matrix:
[[22698  1152]
 [  258  6543]]

--- KNN ---
Accuracy : 0.9056
Precision: 0.8125
Recall   : 0.7472
F1 Score : 0.7785
Confusion Matrix:
[[22677  1173]
 [ 1719  5082]]

--- Decision Tree ---
Accuracy : 0.9869
Precision: 0.9700
Recall   : 0.9707
F1 Score : 0.9704
Confusion Matrix:
[[23646   204]
 [  199  6602]]

--- Random Forest ---
Accuracy : 0.9847
Precision: 0.9490
Recall   : 0.9841
F1 Score : 0.9662
Confusion Matrix:
[[23490   360]
 [  108  6693]]

--- SVM ---
Accuracy : 0.9732
Precision: 0.8954
Recall   : 0.9956
F1 Score : 0.9428
Confusion Matrix:
[[23059   791]
 [   30  6771]]

--- Naive Bayes ---
Accuracy : 0.8543
Precision: 0.6142
Recall   : 0.9234
F1 Score : 0.7377
Confusion Matrix:
[[19905  3945]
 [  521  6280]]

--- Gradient Boosting ---
Accuracy : 0.9886
Precision: 0.9801
Recall   : 0.9685
F1 Score : 0.9743
Confusion Matrix:
[[23716   134]
 [  214  6587]]

--- 

In [74]:
import pandas as pd

results_df = pd.DataFrame(results)

best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]

best_model = fitted_models[best_model_name]

print("Best Model:", best_model_name)
print("Best F1 Score:",
      results_df["F1 Score"].max())

Best Model: XGBoost
Best F1 Score: 0.9889592227292802


In [75]:
import joblib

joblib.dump(best_model, "best_classification_model.pkl")
joblib.dump(scaler, "classification_scaler.pkl")


print("Model saved successfully!")

Model saved successfully!


In [76]:
model_columns = X_c.columns.tolist()
joblib.dump(model_columns, "model_columns_class.pkl")

['model_columns_class.pkl']